# Stage 2: Continue-pretrain (ветка Math&Code, 75:15:10)

Зеркало ветки Math&Code из TinyLlama v1.1: модель продолжает предобучение из
лучшего чекпойнта `1.pretrain` на смеси 75% NL + 15% code + 10% math
(в статье: 75% SlimPajama + 15% StarCoder + 10% Proof Pile 2).

Архитектура обязана совпадать 1-в-1 с pretrain (тот же `model.py`), иначе
загрузка весов упадёт. Кэши блоков из `../data/` переиспользуются — меняется
только взвешенный микс.

Запуск: cwd = `2.continue_pretrain`. Указать `PRETRAIN_DIR` на собранный
pretrain-чекпойнт (по умолчанию `../1.pretrain/checkpoints/best`).

## Отклонения от статьи

См. Stage 1 (pretrain) — архитектура и данные идентичны.
Stage 2 отличается только миксом данных и LR.

## 0. Настройки

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd()
EXPERIMENT_ROOT = ROOT.parent
sys.path.insert(0, str(EXPERIMENT_ROOT))  # tinyllama/

DATA_DIR = EXPERIMENT_ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
PRETRAIN_DIR = EXPERIMENT_ROOT / "1.pretrain" / "checkpoints" / "best"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE} | pretrain: {PRETRAIN_DIR}")

SEED = 0
torch.manual_seed(SEED)

# --- данные: 75% NL + 15% code + 10% math (Math&Code branch) ---
MIX = "continue_pretrain"
BLOCK_SIZE = 512
TOTAL_BLOCKS = 250000

# --- тренер: continue-pretrain, мягче, чем с нуля ---
EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 200
MIN_LR = 1e-5
EVAL_STEPS = 500
SAVE_STEPS = 500

# --- MLflow ---
MLFLOW_TRACKING_URI = "http://host.docker.internal:5000"
MLFLOW_EXPERIMENT_NAME = "tinyllama-continue-pretrain"

torch.set_float32_matmul_precision("medium")

assert PRETRAIN_DIR.exists(), f"pretrain checkpoint not found: {PRETRAIN_DIR}"

## 1. Данные: тот же корпус, новый микс (добавилась математика)

In [ ]:
from data import build_eval_dataset, load_tokenizer, make_mixed_dataset

tokenizer = load_tokenizer(DATA_DIR / "tokenizer")

train_ds = make_mixed_dataset(
    MIX,
    DATA_DIR / "blocks",
    tokenizer,
    BLOCK_SIZE,
    total_blocks=TOTAL_BLOCKS,
    seed=SEED,
)
val_ds = build_eval_dataset(tokenizer, BLOCK_SIZE, cache_dir=DATA_DIR / "blocks")
print(f"train blocks={len(train_ds):,} | eval blocks={len(val_ds):,}")

## 2. Модель из pretrain-чекпойнта

In [ ]:
from model import CausalLM

model = CausalLM.from_pretrained(PRETRAIN_DIR)
x = torch.randint(0, model.config.vocab_size, (2, model.config.block_size))
with torch.no_grad():
    loss = model(input_ids=x, labels=x).loss
print(f"ckpt: {PRETRAIN_DIR} | params: {model.num_params():,} | fwd ok | loss={loss.item():.3f}")

## 3. HF Trainer

In [ ]:
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, Trainer, TrainingArguments

import mlflow
from train import SampleTextCallback, report_to_value

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": MIN_LR},
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    bf16=(DEVICE == "cuda"),
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    torch_compile=(DEVICE == "cuda"),
    seed=SEED,
    report_to=report_to_value(MLFLOW_TRACKING_URI),
    run_name=MLFLOW_EXPERIMENT_NAME,
    dataloader_pin_memory=False,
)

sample_cb = SampleTextCallback(
    model,
    tokenizer,
    prompts=["The history of Rome", "Machine learning is", "def fibonacci(n):"],
    max_new_tokens=64,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[sample_cb, EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## 4. Проверка

In [ ]:
from train import evaluate_ppl, generate

model = trainer.model
trainer.save_model(str(CKPT_DIR / "best"))
print("best ckpt:", CKPT_DIR / "best")

ppl = evaluate_ppl(model, val_ds, tokenizer, device=DEVICE)
print(f"val perplexity: {ppl:.2f}")

print("\nsample (text):")
print(generate(model, tokenizer, "The history of Rome", max_new_tokens=48))
print("\nsample (code):")
print(generate(model, tokenizer, "def fibonacci(n):", max_new_tokens=48))